# Weekly project 6
Today we will continue work from monday.
We will follow the style of last week.

Weekly project:
- You will need to implement your own k-means algorithm. You are not allowed to use the implementation in `sklearn`.
- It should be able to cluster each of the different figures.
- Extend your k-means so it finds the optimal amount of clusters.
  
## Challenge
- Implement the mean shift clustering algorithm


In [18]:
import numpy as np
import open3d as o3d
import copy
import matplotlib.pyplot as plt

def draw_labels_on_model(pcl, labels):
    cmap = plt.get_cmap("tab20")
    pcl_temp = copy.deepcopy(pcl)
    max_label = labels.max()
    colors = cmap(labels / (max_label if max_label > 0 else 1))
    colors[labels < 0] = 0
    pcl_temp.colors = o3d.utility.Vector3dVector(colors[:, :3])
    o3d.visualization.draw_geometries([pcl_temp])

d = 4
mesh = o3d.geometry.TriangleMesh.create_tetrahedron().translate((-d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_octahedron().translate((0, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_icosahedron().translate((d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_torus().translate((-d, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_mobius(twists=1).translate((0, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_mobius(twists=2).translate((d, -d, 0))

## apply k means on this
point_cloud = mesh.sample_points_uniformly(int(1e3))

labels=kmeans(6,point_cloud)
draw_labels_on_model(point_cloud,labels)


In [ ]:
def kmeans(k: int, pcl:o3d.geometry.PointCloud): # arg1= k  arg2=point_cloud
    """Your implementation goes here

    Args:
    Replace the arguments by what you think is necessary.
        
        pcl (_o3d.geometry.PointCloud_): Point cloud to visualize
        k (_int_): k clusters
    """

    max_iter=300


    centroid_threshold=0.01

    

    convergence = False

    points = np.asarray(pcl.points)

    # random initialization of k clusters
    random_indices = np.random.choice(len(points),size=k, replace=False) # select k amount of integers from 0-points_max_index randomly
    prev_centroid_list=points[random_indices].copy()

    centroid_list= np.zeros((k,3))# np array verison of [[0,0,0] for _ in range(k)]


    iter=0
    while(not convergence and iter<max_iter): # if one of these becomes false, then the loop stops

        # reset every iteration
        point_cluster_pair=[[] for _ in range(k)]

        labels=[]

        # find closest centroid to each point 
        for point in points:
            min_diff=np.inf
            closest_centroid_label=None
            for i,centroid in enumerate(prev_centroid_list):
                diff = np.linalg.norm(point - centroid) # computing euclidean distance
                if (diff<min_diff):
                    closest_centroid_label=i
                    min_diff=diff

            point_cluster_pair[closest_centroid_label].append((point)) # will have the same index as the centeroid list
            labels.append(closest_centroid_label)

        #recompute each cluster centre: 
        for i,cluster in enumerate(point_cluster_pair):

            if len(cluster)>0: # making sure the cluster is not empty
                centroid_list[i]=np.mean(cluster,axis=0)
            else:
                centroid_list[i]=prev_centroid_list[i]


        # check convergence 
        conv_count=0

        for i,centroid in enumerate(centroid_list):
            if (np.linalg.norm(centroid-prev_centroid_list[i])<centroid_threshold):
                conv_count+=1

        # if convergence is met
        if conv_count==k:
            convergence=True

        # update for next iteration
        iter+=1

        prev_centroid_list=centroid_list.copy()

    # getting the most recent label
    labels = []

    for point in points:
        distances = np.linalg.norm(centroid_list - point, axis=1)
        labels.append(np.argmin(distances)) # argmin returns the index of closest centeroid (smallest distance)



    return np.array(labels) # label size = pcl points size and order is preserved so these labels can directly be mapped to the pcl points. 

        






